In [ ]:
import random
import copy
import os
import torch
from run_cv import about_data
from utils_sampling_processing import make_neg_edges
import shutil
import sys
import pandas as pd
import numpy as np


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
node_type1 = 'drug'
node_type2 = 'disease'
rel = 'indication'

config = {
    "node_type1": node_type1,
    "node_type2": node_type2,
    "rel": rel,
    "seeded_type": 'disease',
    "data_path": '../data_testy/no_merge',
    "out_path": '../out_testy/no_merge'
}

# data_path = '../data_testy/no_merge/hetero_graph.pt'
# data = torch.load(data_path)


In [ ]:
def process_batch(batch, config, stage = 'train'): #, ind_MSG_Pass = True):
    node_type1 = config['node_type1']
    node_type2 = config['node_type2']
    rel = config['rel']

    # Extract the edge index and labels
    labeled = 'edge_label' in batch[(node_type1, rel, node_type2)]

    # Use the mask to filter the edge index and labels
    processed_edge_types = set()
    for edge_type in batch.edge_types:
        # if edge_type[0] == 'disease' or edge_type[2] == 'disease':
        if edge_type[1] in processed_edge_types:
            continue
        else:
            processed_edge_types.add(edge_type[1])
            # print(f"Processing edge type: {edge_type}")

            if edge_type[0] == config['seeded_type']: # and edge_type[1] == rel:
                continue

            edge_index = batch[edge_type].edge_index
            edge_mask = batch[edge_type].mask
            # print(f"\tEdge index shape: {edge_index.shape}, Edge mask shape: {edge_mask.shape}")

            if stage == 'train':
                mask = edge_mask == 0
                if edge_type[1] == rel:
                    print('training, mask len:', sum(mask))
            elif stage == 'val':
                mask = edge_mask != 2
                if edge_type[1] == rel:
                    print('validating, mask len:', sum(mask))
            elif stage == 'test':
                mask = torch.ones(edge_index.shape[1], dtype=torch.bool)
                if edge_type[1] == rel:
                    print('testing, mask len:', sum(mask))
            else:
                raise ValueError(f"Invalid data stage mask type: {stage}. Must be 'train', 'val', or 'test'.")

            edge_index = edge_index[:, mask]
            edge_mask = edge_mask[mask]
            # print(f"Filtered edge index shape: {edge_index.shape}, Filtered edge mask shape: {edge_mask.shape}")

            if edge_type[1] == rel:

                if labeled:
                    edge_label = batch[edge_type].edge_label
                    edge_label = edge_label[mask]

                    # if 0 in edge_label:
                    #     indications = indications[:, edge_label != 0] 

                # if config['ind_MSG_Pass']:
                batch_index = (edge_index[1] < config['batch_size'])
                print('batch_index size:', batch_index.sum())
                # edges_index[0] or [1] depending on which entry is used for conditional edge_type[0]/[1] == config['seeded_type']
                # else:
                #     batch_index = (edge_index[1] <= edge_index[1].max())

                # print(f"Batch index shape: {batch_index.shape}, sum: {batch_index.sum().item()} ({(batch_index.sum().item()/batch_index.shape[0])*100:.2f}%)")

                if config['seeded_ind_only']:
                    edge_index = edge_index[:, batch_index]
                    edge_label = edge_label[batch_index] if labeled else None
                    indications = edge_index
                else:
                    edges_batch = edge_index[:, batch_index]
                    labels_batch = edge_label[batch_index] if labeled else None
                    indications = edges_batch

                    edge_index = edge_index[:, ~batch_index]
                    edge_label = edge_label[~batch_index] if labeled else None

                if stage == 'train':
                    num_batch = indications.shape[1]
                    target_batch = torch.zeros(num_batch, dtype=torch.bool)
                    targets = random.sample(range(num_batch), int(num_batch*config['sup_split']))
                    target_batch[targets] = True

                    if not config['seeded_ind_only']:
                        num_index = edge_index.shape[1]
                        index_MSGPASS = torch.zeros(num_index, dtype=torch.bool)
                        MSGPASS = random.sample(range(num_index), int(num_index*(1 - config['sup_split'])))
                        index_MSGPASS[MSGPASS] = True
                        print('non-batch for MSG-Pass:', index_MSGPASS.sum())

                        edge_index = edge_index[:, index_MSGPASS]
                        edge_label = edge_label[index_MSGPASS] if labeled else None

                else: 
                    mask_batch = edge_mask[batch_index]
                    target_batch = mask_batch == 1 if stage == 'val' or 2 not in mask_batch else mask_batch == 2

                print('targets:', target_batch.sum())

                if config['seeded_ind_only']:
                    edges_batch = edge_index[:, target_batch]
                    labels_batch = edge_label[target_batch] if labeled else None
                    edge_index = edge_index[:, ~target_batch]
                    edge_label = edge_label[~target_batch] if labeled else None

                else:
                    add_to_index = edges_batch[:, ~target_batch]
                    add_to_labels = labels_batch[~target_batch] if labeled else None

                    edges_batch = edges_batch[:, target_batch]
                    labels_batch = labels_batch[target_batch] if labeled else None

                    edge_index = torch.cat((edge_index, add_to_index), dim=1)
                    edge_label = torch.cat((edge_label, add_to_labels)) if labeled else None
                    # print(f"added {add_to_index.shape[1]} edges and labels for MSG-PASS")

                if labeled:
                    if 0 in edge_label:
                        edge_index = edge_index[:, edge_label != 0] 
                        edge_label = edge_label[edge_label != 0]

                    # Not really needed, but for consistency
                    batch[edge_type].edge_label = edge_label 
                    batch[(edge_type[2], edge_type[1], edge_type[0])].edge_label = edge_label


                # print(f"edge_index shape: {edge_index.shape}, Labels batch shape: {edge_label.shape if labeled else 'N/A'}")
                # batch[(edge_type[2], edge_type[1], edge_type[0])].edge_index = edge_index.flip(0)

            batch[edge_type].edge_index = edge_index
            batch[(edge_type[2], edge_type[1], edge_type[0])].edge_index = edge_index.flip(0)
            # print(f"Updated edge index shape: {batch[edge_type].edge_index.shape}\n")

    if labeled:
        if 2 in labels_batch and config['balanced']:
            # Make edges_batch balanced btwn contraindications and indications; do not remove from edge_index
            contras = edges_batch[:, labels_batch == 2]
            contras = contras[:,random.sample(range(contras.shape[1]), (labels_batch == 1).sum().item())]

            edges_batch = torch.cat((edges_batch[:, labels_batch != 2], contras), dim=1)
            labels_batch = torch.cat((labels_batch[labels_batch != 2], torch.full((contras.shape[1],), 2)))

        num_negs = (labels_batch == 1).sum().item() if 0 not in labels_batch else None

    elif not labeled:
        labels_batch = torch.ones(edges_batch.shape[1])
        num_negs = edges_batch.shape[1]

    if 0 not in labels_batch:
        # Generate negative edges
        # neg_edges = make_neg_edges(edges_batch, num_negs)
        neg_edges = make_neg_edges(indications, num_negs)
        # could generate an edge that exists but is not present on edges_batch as a neg edge... what's the probably that sampled drug/disease pair will not have a sampled indication? 
        # for validation mask == 1, much likely an existing ind is not in edges_batch

        # print(neg_edges.shape, '\tneg_edges shape:')
        # print(edges_batch.shape, '\tedges_batch shape before')
        # print(edge_label.shape, '\tlabels_batch shape before')
        # print(edge_label.unique(), '\tlabels_batch unique before')

        labels_batch = torch.cat((labels_batch, torch.zeros(neg_edges.shape[1])))
        edges_batch = torch.cat((edges_batch, neg_edges), dim=1)

    # print(edges_batch.shape, '\tedges_batch shape after')
    # print(labels_batch.shape, '\tlabels_batch shape after')
    # print(labels_batch.unique(), '\tlabels_batch unique after')

    # Shuffle the indices 
    shuffle_indices = torch.randperm(labels_batch.size(0))
    labels_batch = labels_batch[shuffle_indices]
    edges_batch = edges_batch[:, shuffle_indices]


#    print('batch size (edges):', labels_batch.shape)
#    print('real edges in batch:', batch[(node_type1, rel, node_type2)].edge_index.shape)

    return batch, edges_batch, labels_batch


In [ ]:
from torch_geometric.loader import HGTLoader

def sampling(config, data, workers = 0):
    loader = HGTLoader(data, 
                   num_samples=[config['num_samples']] * config['sampling_layers'], 
                   batch_size=config['batch_size'], 
                #    input_nodes=(config['node_type1'], data[config['node_type1']]['input_nodes']), 
                   input_nodes=(config['seeded_type'], data[config['seeded_type']].input_nodes) if 'input_nodes' in data[config['seeded_type']] else (config['seeded_type'], None), 
                   shuffle=True, 
                   num_workers = workers, 
                   persistent_workers = bool(workers)
                   )
    return loader


In [ ]:
# from utils_sampling_processing import sampling

config['num_samples'] = 512
config['sampling_layers'] = 3
config['batch_size'] = 273

config['seeded_ind_only'] = True
config['sup_split'] = 0.8

config['data_path'] = '../data_testy/no_merge'
data_CV = torch.load(config['data_path'] + '/CV1/data.pt')
loader = sampling(config, data_CV, 0)

In [ ]:
def NumOfEdges(data, data2 = None):
    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] not in processed_edge_types:
            processed_edge_types.add(edge_type[1])

            if data2 is None:
                print(f'{edge_type[1]}: {data[edge_type].num_edges}')
            else:
                dif = data[edge_type].num_edges - data2[edge_type].num_edges
                if dif != 0:
                    print(f'{edge_type[1]} dif: {dif}')

NumOfEdges(data_CV)


In [ ]:
data_endos = torch.load(config['data_path'] + '/hetero_graph_endos.pt')
data_iAT = torch.load(config['data_path'] + '/hetero_graph_AT.pt')
data_MIRs = torch.load(config['data_path'] + '/hetero_graph_MIRs.pt')

In [ ]:
NumOfEdges(data_CV, data_endos)

In [ ]:
NumOfEdges(data_CV, data_iAT)

In [ ]:
NumOfEdges(data_CV, data_MIRs)

In [ ]:
data_CV['disease'].input_nodes.sum(), data_CV['disease'].num_nodes

In [ ]:
for i in range(2, 16):
    batch_size = 1363 / i
    print(f'{i}: {batch_size:.2f}, {17080 / batch_size:.2f}')

In [ ]:
17080 / 341

In [ ]:
config

In [ ]:
len(loader)

In [ ]:
def batch_edges(loader):
    edges = []
    for i, batch in enumerate(iter(loader)):
        print(i, ':', batch.num_edges)

        processed_edge_types = set()
        edges_in_batch = {}
        for edge_type in batch.edge_types:
            if edge_type[1] not in processed_edge_types:
                processed_edge_types.add(edge_type[1])

                # print(f'{edge_type[1]}: {data[edge_type].num_edges}')
                edges_in_batch[edge_type[1]] = batch[edge_type].num_edges

        edges.append(edges_in_batch)
    return pd.DataFrame(edges).agg(['mean', 'std']).T

batch_edges(loader)

In [ ]:
config['batch_size'] = 1364 // 4 + 1
loader = sampling(config, data_CV, 0)

means = batch_edges(loader)
print(means)

In [ ]:
means = pd.concat([means, batch_edges(sampling(config, data_endos, 0)), batch_edges(sampling(config, data_iAT, 0)), batch_edges(sampling(config, data_MIRs, 0))], axis=1)

means

In [ ]:
means.columns = ['mean', 'std', 'endos_mean', 'endos_std', 'iAT_mean', 'iAT_std', 'MIRs_mean', 'MIRs_std']
means

In [ ]:
differences = means[['mean', 'std']]

differences['endos_mean_dif'] = (means['mean'] - means['endos_mean']) / means['mean'] * 100
differences['endos_std_dif'] = (means['std'] - means['endos_std']) / means['std'] * 100
differences['iAT_mean_dif'] = (means['mean'] - means['iAT_mean']) / means['mean'] * 100
differences['iAT_std_dif'] = (means['std'] - means['iAT_std']) / means['std'] * 100
differences['MIRs_mean_dif'] = (means['mean'] - means['MIRs_mean']) / means['mean'] * 100
differences['MIRs_std_dif'] = (means['std'] - means['MIRs_std']) / means['std'] * 100

differences = differences.round(2)

In [ ]:
differences[['mean', 'endos_mean_dif', 'iAT_mean_dif', 'MIRs_mean_dif']]

In [ ]:
differences[['mean', 'endos_std_dif', 'iAT_std_dif', 'MIRs_std_dif']]

In [ ]:
means['dif'] = (means['mean'] - means['endos_mean']) / means['mean'] * 100
means[['mean', 'endos_mean', 'dif']].round(2)

In [ ]:
config['batch_size'] = 134
loader = sampling(config, data_CV, 0)

means = batch_edges(loader)
means = pd.concat([means, batch_edges(sampling(config, data_endos, 0)), batch_edges(sampling(config, data_iAT, 0)), batch_edges(sampling(config, data_MIRs, 0))], axis=1)

means.columns = ['mean', 'std', 'endos_mean', 'endos_std', 'iAT_mean', 'iAT_std', 'MIRs_mean', 'MIRs_std']

differences = means[['mean', 'std']]

differences['endos_mean_dif'] = (means['mean'] - means['endos_mean']) / means['mean'] * 100
differences['endos_std_dif'] = (means['std'] - means['endos_std']) / means['std'] * 100
differences['iAT_mean_dif'] = (means['mean'] - means['iAT_mean']) / means['mean'] * 100
differences['iAT_std_dif'] = (means['std'] - means['iAT_std']) / means['std'] * 100
differences['MIRs_mean_dif'] = (means['mean'] - means['MIRs_mean']) / means['mean'] * 100
differences['MIRs_std_dif'] = (means['std'] - means['MIRs_std']) / means['std'] * 100

differences = differences.round(2)

In [ ]:
differences[['mean', 'endos_mean_dif', 'iAT_mean_dif', 'MIRs_mean_dif']]

In [ ]:
differences[['std', 'endos_std_dif', 'iAT_std_dif', 'MIRs_std_dif']]

In [ ]:
from torch_geometric.utils import degree

def avg_degrees(data):
    print(f"Average degree of the entire graph: {data.num_edges / data.num_nodes / 2:.2f} \n")

    for node_type in data.node_types:
        # Count edges involving this node type
        total_edges = sum(
            data[edge_type].edge_index.size(1)
            for edge_type in data.edge_types
            if edge_type[0] == node_type
        )
        num_nodes = data[node_type].num_nodes
        average_degree = total_edges / num_nodes
        print(f"Average degree for node type '{node_type}' ({num_nodes} total): \t{average_degree:.2f}")

    print()

    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] not in processed_edge_types:
            processed_edge_types.add(edge_type[1])

            num_edges = data[edge_type].num_edges
            edge_index = data[edge_type].edge_index

            source_nodes = edge_index[0].unique().size(0)
            target_nodes = edge_index[1].unique().size(0)

            average_degree_source = num_edges / source_nodes
            average_degree_target = num_edges / target_nodes
            
            src_degrees = degree(edge_index[0], num_nodes=data[edge_type[0]].num_nodes)
            dst_degrees = degree(edge_index[1], num_nodes=data[edge_type[2]].num_nodes)

            print(f"Edge type '{edge_type[1]}':")
            print(f"\tAverage degree for {edge_type[0]} ({source_nodes}): {average_degree_source:.2f} (max deg: {src_degrees.max()})")
            print(f"\tAverage degree for {edge_type[2]} ({target_nodes}): {average_degree_target:.2f} (max deg: {dst_degrees.max()})")

In [ ]:
node_type1, rel, node_type2

In [ ]:
from torch_geometric.utils import degree

src_degrees = degree(data_CV[(node_type1, rel, node_type2)].edge_index[0], num_nodes=data_CV[node_type1].num_nodes)

src_degrees.unique().shape, sum(src_degrees != 0), data_CV[(node_type1, rel, node_type2)].edge_index[0].unique().shape, data_CV[(node_type1, rel, node_type2)].edge_index[1].unique().shape

In [ ]:
avg_degrees(data_CV)

In [ ]:
avg_degrees(data_endos)

In [ ]:
avg_degrees(data_iAT)

In [ ]:
avg_degrees(data_MIRs)

In [ ]:
def batch_edges(batch_size, num_samples, data):
    config['batch_size'] = batch_size
    config['num_samples'] = num_samples

    loader = sampling(config, data, 0)

    edges = []
    for i, batch in enumerate(iter(loader)):
        # print(i, ':', batch.num_edges)

        processed_edge_types = set()
        edges_in_batch = {}
        # edges_in_batch['batch_size'] = batch_size
        # edges_in_batch['num_samples'] = num_samples
        for edge_type in batch.edge_types:
            if edge_type[1] not in processed_edge_types:
                processed_edge_types.add(edge_type[1])

                # print(f'{edge_type[1]}: {data[edge_type].num_edges}')
                edges_in_batch[edge_type[1]] = batch[edge_type].num_edges

        edges.append(edges_in_batch)
        df = pd.DataFrame(edges)
    return df.agg(['mean']).round(2), df.agg(['std']).round(2)#.T

batch_edges(1364 // 4 + 1, 512, data_CV)

5k              prot_prot
[2k]            [drug_drug]
[2k]    phen    [dis_phen_pos]
[1.5]           [dis_dis]
1194            drug_drug
1156    prot    dis_prot
[1k]    eff     [drug_eff]
[971]   prot    [drug_prot]
744     eff     drug_eff
[654]   dis     [contra]
[(625)]         [drug_drug]
596     prot    drug_prot
[503]    dis    [dis_phen_pos]
403     dis     dis_phen_pos
338     drug    drug_eff
306     drug    drug_prot
304     dis     contra
179     dis     dis_prot
148     drug    ind
138     phen    dis_phen_pos
108     drug    contra
103     dis     ind
75      dis     exp_dis
(63)    drug    drug_eff
59      drug    off-label
(54)    eff     drug_eff
(53)    prot    dis_prot
44      exp     exp_dis
(35)            prot_prot
<= 21   x4      off, dis_phen_neg, dis_dis

In [ ]:
import matplotlib.pyplot as plt

def plot(df, columns_to_plot = ['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein']):
    num_columns = len(columns_to_plot)

    # for layer in df['Layer'].unique():
        # df_layer = df[df['Layer'] == layer]

    fig, axes = plt.subplots(1, num_columns, figsize=(30, 5), sharey=True)
    # fig.suptitle(f'Layer {layer}: Metrics vs Size (Grouped by Sample)', fontsize=16)

    markers = ['o', 's', 'D', '^', 'v', '<', '>']
    for i, column in enumerate(columns_to_plot):
        ax = axes[i]
        for size, group in df.groupby('batch_size'):
            marker = markers[int(size) % len(markers)]
            ax.plot(group['num_samples'], group[column], marker=marker, label=f'size {size}')
        ax.set_title(column)
        ax.set_xlabel('num_samples')
        if i == 0:
            ax.set_ylabel('Value')
        ax.grid(True)

    # Add a single legend for the entire figure
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, title='Size', bbox_to_anchor=(.85, 0.5), loc='center left')

    plt.tight_layout(rect=[0, 0, 0.85, 0.95])
    plt.show()



In [ ]:
def get_diff(sizes, samples, data):
    means, stds = pd.DataFrame(), pd.DataFrame()
    for sample in samples:
        for size in sizes:
            print(f'size = {size}, sample = {sample}')
            
            batch_means, batch_stds = batch_edges(size, sample, data_CV)# - batch_edges(size, sample, data_endos)
            batch_means_endos, batch_stds_endos = batch_edges(size, sample, data)# - batch_edges(size, sample, data_endos)

            mean_dif = (batch_means - batch_means_endos) / batch_means * 100
            std_dif = (batch_stds - batch_stds_endos) / batch_stds * 100

            mean_dif['batch_size'] = size
            std_dif['batch_size'] = size
            mean_dif['num_samples'] = sample
            std_dif['num_samples'] = sample

            means = pd.concat([means, mean_dif], ignore_index=True)
            stds = pd.concat([stds, std_dif])

    return means, stds



In [ ]:
samples = [35, 44, 50, 53, 54, 59, 63, 75, 103, 108, 138, 148, 179, 250, 306, 338, 350, 403, 450, 512, 550, 596, 654, 971, 1200]
sizes = [134, 342]

means_diff_endos, stds_diff_endos = get_diff(sizes, samples, data_endos)
plot(means_diff_endos, columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

In [ ]:
plot(means_diff_endos[0], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

In [ ]:
means_diff_iAT, stds_diff_iAT = get_diff(sizes, samples, data_iAT)
plot(means_diff_iAT, columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])


In [ ]:
means_diff_MIRs, stds_diff_MIRs = get_diff(sizes, samples, data_MIRs)
plot(means_diff_MIRs, columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

In [ ]:
plot(means_diff_endos[means_diff_endos['num_samples'] < 900], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

In [ ]:
print(samples)

In [ ]:
plot(means_diff_iAT[means_diff_iAT['num_samples'] < 900], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

try 450, 512 and 550 for samples, with size 342c

In [ ]:
plot(stds_diff_endos[stds_diff_endos['num_samples'] < 596], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

In [ ]:
plot(stds_diff_iAT[stds_diff_iAT['num_samples'] < 596], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

# Trimmed sampling

In [ ]:
from run_cv import about_data
from utils_sampling_processing import make_neg_edges, process_batch
import pandas as pd
import numpy as np
import torch


In [ ]:
data_trimmed = torch.load('../data_trimmed/CV1/data.pt')

In [ ]:
from torch_geometric.loader import HGTLoader

def sampling(config, data, workers = 0):
    loader = HGTLoader(data, 
                   num_samples=[config['num_samples']] * config['sampling_layers'], 
                   batch_size=config['batch_size'], 
                #    input_nodes=(config['node_type1'], data[config['node_type1']]['input_nodes']), 
                   input_nodes=(config['seeded_type'], data[config['seeded_type']].input_nodes) if 'input_nodes' in data[config['seeded_type']] else (config['seeded_type'], None), 
                   shuffle=True, 
                   num_workers = workers, 
                   persistent_workers = bool(workers)
                   )
    return loader


In [ ]:
def NumOfEdges(data, data2 = None):
    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] not in processed_edge_types:
            processed_edge_types.add(edge_type[1])

            if data2 is None:
                print(f'{edge_type[1]}: {data[edge_type].num_edges}')
            else:
                dif = data[edge_type].num_edges - data2[edge_type].num_edges
                if dif != 0:
                    print(f'{edge_type[1]} dif: {dif}')

NumOfEdges(data_trimmed)


In [ ]:
from torch_geometric.utils import degree

def avg_degrees(data):
    print(f"Average degree of the entire graph: {data.num_edges / data.num_nodes / 2:.2f} \n")

    for node_type in data.node_types:
        # Count edges involving this node type
        total_edges = sum(
            data[edge_type].edge_index.size(1)
            for edge_type in data.edge_types
            if edge_type[0] == node_type
        )
        num_nodes = data[node_type].num_nodes
        average_degree = total_edges / num_nodes
        print(f"Average degree for node type '{node_type}' ({num_nodes} total): \t{average_degree:.2f}")

    print()

    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] not in processed_edge_types:
            processed_edge_types.add(edge_type[1])

            num_edges = data[edge_type].num_edges
            edge_index = data[edge_type].edge_index

            source_nodes = edge_index[0].unique().size(0)
            target_nodes = edge_index[1].unique().size(0)

            average_degree_source = num_edges / source_nodes
            average_degree_target = num_edges / target_nodes
            
            src_degrees = degree(edge_index[0], num_nodes=data[edge_type[0]].num_nodes)
            dst_degrees = degree(edge_index[1], num_nodes=data[edge_type[2]].num_nodes)

            print(f"Edge type '{edge_type[1]}':")
            print(f"\tAverage degree for {edge_type[0]} ({source_nodes}): {average_degree_source:.2f} (max deg: {src_degrees.max()})")
            print(f"\tAverage degree for {edge_type[2]} ({target_nodes}): {average_degree_target:.2f} (max deg: {dst_degrees.max()})")

In [ ]:
avg_degrees(data_trimmed)

In [ ]:
def batch_edges(batch_size, num_samples, data, config = {'sampling_layers': 3, 'seeded_type': 'disease'}):
    config['batch_size'] = batch_size
    config['num_samples'] = num_samples

    loader = sampling(config, data, 0)

    edges = []
    for i, batch in enumerate(iter(loader)):
        # print(i, ':', batch.num_edges)

        processed_edge_types = set()
        edges_in_batch = {}
        # edges_in_batch['batch_size'] = batch_size
        # edges_in_batch['num_samples'] = num_samples
        for edge_type in batch.edge_types:
            if edge_type[1] not in processed_edge_types:
                processed_edge_types.add(edge_type[1])

                # print(f'{edge_type[1]}: {data[edge_type].num_edges}')
                edges_in_batch[edge_type[1]] = batch[edge_type].num_edges

        edges.append(edges_in_batch)
        df = pd.DataFrame(edges)
    return df.agg(['mean']).round(2), df.agg(['std']).round(2)#.T



In [ ]:
batch_edges(1364 // 4 + 1, 512, data_trimmed)[0]

In [ ]:
import matplotlib.pyplot as plt

def plot(df, columns_to_plot = ['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein']):
    num_columns = len(columns_to_plot)

    # for layer in df['Layer'].unique():
        # df_layer = df[df['Layer'] == layer]

    fig, axes = plt.subplots(1, num_columns, figsize=(30, 5), sharey=True)
    # fig.suptitle(f'Layer {layer}: Metrics vs Size (Grouped by Sample)', fontsize=16)

    markers = ['o', 's', 'D', '^', 'v', '<', '>']
    for i, column in enumerate(columns_to_plot):
        ax = axes[i]
        for size, group in df.groupby('batch_size'):
            marker = markers[int(size) % len(markers)]
            ax.plot(group['num_samples'], group[column], marker=marker, label=f'size {size}')
        ax.set_title(column)
        ax.set_xlabel('num_samples')
        if i == 0:
            ax.set_ylabel('Value')
        ax.grid(True)

    # Add a single legend for the entire figure
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, title='Size', bbox_to_anchor=(.85, 0.5), loc='center left')

    plt.tight_layout(rect=[0, 0, 0.85, 0.95])
    plt.show()



In [ ]:
def get_diff(sizes, samples, data_ref, data):
    means, stds = pd.DataFrame(), pd.DataFrame()
    for sample in samples:
        for size in sizes:
            print(f'size = {size}, sample = {sample}')

            batch_means, batch_stds = batch_edges(size, sample, data_ref)# - batch_edges(size, sample, data_endos)
            batch_means_endos, batch_stds_endos = batch_edges(size, sample, data)# - batch_edges(size, sample, data_endos)

            mean_dif = (batch_means - batch_means_endos) / batch_means * 100
            std_dif = (batch_stds - batch_stds_endos) / batch_stds * 100

            mean_dif['batch_size'] = size
            std_dif['batch_size'] = size
            mean_dif['num_samples'] = sample
            std_dif['num_samples'] = sample

            means = pd.concat([means, mean_dif], ignore_index=True)
            stds = pd.concat([stds, std_dif])

    return means, stds



In [ ]:
data__trimmed_AT = torch.load('../data_trimmed/hetero_graph_AT.pt')

In [ ]:
1364 // 4 + 1

In [ ]:
sizes = [134, 342]
samples = [35, 54, 63, 108, 138, 148, 179, 218, 306, 338, 403, 473, 512, 596, 744, 1200, 5200]

In [ ]:
means_diff_AT, stds_diff_AT = get_diff(sizes, samples, data_trimmed, data__trimmed_AT)
plot(means_diff_AT, columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive'])

In [ ]:
plot(means_diff_AT[means_diff_AT['num_samples'] < 1200], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive', 'phenotype_protein'])


In [ ]:
data_full = torch.load('../data_testy/no_merge/CV1/data.pt')
means_diff_full, stds_diff_full = get_diff(sizes, samples, data_trimmed, data_full)
plot(means_diff_full, columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive', 'phenotype_protein'])

In [ ]:
data_trimmed_semi = torch.load('../data_trimmed_semi/CV1/data.pt')
means_diff_trimmed_semi, stds_diff_trimmed_semi = get_diff(sizes, samples, data_full, data_trimmed_semi)
plot(means_diff_trimmed_semi, columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive', 'phenotype_protein'])

In [ ]:
plot(means_diff_full[means_diff_full['num_samples'] < 1200], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive', 'phenotype_protein'])

In [ ]:
plot(means_diff_trimmed_semi[means_diff_trimmed_semi['num_samples'] < 1200], columns_to_plot=['indication', 'contraindication', 'off-label use', 'drug_protein', 'disease_protein', 'drug_drug', 'disease_disease', 'protein_protein', 'drug_effect', 'disease_phenotype_positive', 'phenotype_protein'])